# Capstone I: Mutual Fund Analytics
## Notebook 1: Data Ingestion and Validation (Day 1)

**Goal:** load all 10 datasets, inspect their structure, understand the fund master, and validate that every AMFI code is consistent across tables. (The scripts `data_ingestion.py` and `live_nav_fetch.py` do the same job and the live API download outside the notebook.)

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)


def find_processed():
    """Works from notebooks/ (data in ../data/processed), from the project root, or next to the CSVs."""
    for cand in (Path("../data/processed"), Path("data/processed"), Path(".")):
        if cand.exists() and list(cand.glob("*fund_master_clean.csv")):
            return cand
    raise FileNotFoundError(
        "Cleaned CSVs not found. Keep this notebook in notebooks/ (CSVs in ../data/processed) "
        "or put it in the same folder as the *_clean.csv files."
    )


PROC = find_processed()
FIG = (PROC.parent.parent / "reports" / "figures") if PROC.name == "processed" else Path("figures")
FIG.mkdir(parents=True, exist_ok=True)
print("Reading from:", PROC)
print("Figures to  :", FIG)


def load(name, dates=()):
    df = pd.read_csv(PROC / f"{name}_clean.csv")
    for c in dates:
        df[c] = pd.to_datetime(df[c])
    return df


def save_fig(name):
    plt.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight")

Reading from: ../data/processed
Figures to  : ../reports/figures


## 1. Load all 10 datasets

In [2]:
NAMES = ["fund_master", "nav_history", "aum_by_fund_house", "monthly_sip_inflows", "category_inflows",
         "industry_folio_count", "scheme_performance", "investor_transactions", "portfolio_holdings", "benchmark_indices"]
data = {n: load(n) for n in NAMES}
summary = pd.DataFrame({n: {"rows": len(d), "columns": d.shape[1], "missing_cells": int(d.isna().sum().sum()), "duplicate_rows": int(d.duplicated().sum())} for n, d in data.items()}).T
summary

,rows,columns,missing_cells,duplicate_rows
fund_master,40,16,0,0
nav_history,64320,4,0,0
aum_by_fund_house,90,5,0,0
monthly_sip_inflows,48,7,12,0
category_inflows,144,3,0,0
industry_folio_count,21,6,0,0
scheme_performance,40,22,40,0
investor_transactions,32778,20,0,0
portfolio_holdings,322,8,0,0
benchmark_indices,8050,3,0,0


In [3]:
for n, d in data.items():
    print("=" * 70); print(n, d.shape); print(d.dtypes.to_string()); display(d.head(3))

fund_master (40, 16)
amfi_code                int64
fund_house                 str
scheme_name                str
category                   str
sub_category               str
plan                       str
launch_date                str
benchmark                  str
expense_ratio_pct      float64
exit_load_pct          float64
min_sip_amount           int64
min_lumpsum_amount       int64
fund_manager               str
risk_category              str
sebi_category_code         str
expense_ratio_valid      int64


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code,expense_ratio_valid
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.00,500,1000,Sohini Andani,Moderate,EC01,1
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.00,500,1000,Sohini Andani,Moderate,EC01,1
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.00,500,1000,R. Srinivasan,Very High,EC03,1


nav_history (64320, 4)
amfi_code      int64
date             str
nav          float64
is_filled      int64


,amfi_code,date,nav,is_filled
0,100016,2022-01-03,520.46,0
1,100016,2022-01-04,515.10,0
2,100016,2022-01-05,521.72,0


aum_by_fund_house (90, 5)
date                  str
fund_house            str
aum_lakh_crore    float64
aum_crore           int64
num_schemes         int64


...

,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status,is_amount_outlier,scheme_name,fund_house,category,sub_category,plan,month
0,INV000081,2024-01-01,120844,SIP,1842,Gujarat,Surat,T30,36-45,Male,31.00,Net Banking,Verified,False,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra Mutual Fund,Debt,Liquid,Regular,2024-01
1,INV000097,2024-01-01,125498,Redemption,527377,Karnataka,Mysore,B30,18-25,Male,6.00,Net Banking,Verified,False,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Equity,Mid Cap,Direct,2024-01
2,INV000234,2024-01-01,148567,SIP,28094,Tamil Nadu,Coimbatore,B30,26-35,Male,23.80,Mandate,Verified,False,Mirae Asset Large Cap Fund - Regular - Growth,Mirae Asset Mutual Fund,Equity,Large Cap,Regular,2024-01


portfolio_holdings (322, 8)
amfi_code              int64
stock_symbol             str
stock_name               str
sector                   str
weight_pct           float64
market_value_cr      float64
current_price_inr    float64
portfolio_date           str


,amfi_code,stock_symbol,stock_name,sector,weight_pct,market_value_cr,current_price_inr,portfolio_date
0,119551,POWERGRID,Power Grid Corporation,Utilities,13.85,737.09,"6,011.08",2025-12-31
1,119551,HDFCBANK,HDFC Bank Ltd,Banking,11.19,88.97,"1,074.65",2025-12-31
2,119551,GRASIM,Grasim Industries Ltd,Diversified,9.90,208.45,"5,964.59",2025-12-31


benchmark_indices (8050, 3)
date               str
index_name         str
close_value    float64


,date,index_name,close_value
0,2022-01-03,NIFTY50,"17,492.79"
1,2022-01-04,NIFTY50,"17,689.64"
2,2022-01-05,NIFTY50,"17,835.05"


## 2. Understand the fund master

In [4]:
fm = data["fund_master"]
for col in ["fund_house", "category", "sub_category", "plan", "risk_category"]:
    print(f"\n{col} ({fm[col].nunique()} unique):"); print(fm[col].value_counts().to_string())


fund_house (10 unique):
fund_house
SBI Mutual Fund                      5
HDFC Mutual Fund                     5
ICICI Prudential Mutual Fund         5
Nippon India Mutual Fund             5
Kotak Mahindra Mutual Fund           4
Axis Mutual Fund                     4
Aditya Birla Sun Life Mutual Fund    3
UTI Mutual Fund                      3
Mirae Asset Mutual Fund              3
DSP Mutual Fund                      3

category (2 unique):
category
Equity    34
Debt       6

sub_category (12 unique):
sub_category
Large Cap          14
Mid Cap             7
Small Cap           6
Liquid              3
Gilt                2
Flexi Cap           2
Short Duration      1
Value               1
Index/ETF           1
Index               1
Large & Mid Cap     1
ELSS                1

plan (2 unique):
plan
Regular    32
Direct      8

risk_category (5 unique):
risk_category
Moderate           16
High                8
Very High           6
Low                 6
Moderately High     4


## 3. Validate AMFI codes

In [5]:
master = set(fm["amfi_code"])
report = []
for n in ["nav_history", "scheme_performance", "investor_transactions", "portfolio_holdings"]:
    codes = set(data[n]["amfi_code"])
    report.append({"dataset": n, "distinct_codes": len(codes), "codes_not_in_fund_master": len(codes - master), "fund_master_codes_missing_here": len(master - codes)})
report = pd.DataFrame(report)
display(report)
print("Duplicate AMFI codes in fund_master:", int(fm["amfi_code"].duplicated().sum()))
print("PASS: every code in the other tables exists in fund_master." if (report["codes_not_in_fund_master"] == 0).all() else "FAIL: unknown codes found, investigate before continuing.")
print("Note: portfolio_holdings covers equity funds only, so debt funds are expected to be missing there.")

,dataset,distinct_codes,codes_not_in_fund_master,fund_master_codes_missing_here
0,nav_history,40,0,0
1,scheme_performance,40,0,0
2,investor_transactions,40,0,0
3,portfolio_holdings,34,0,6


Duplicate AMFI codes in fund_master: 0
PASS: every code in the other tables exists in fund_master.
Note: portfolio_holdings covers equity funds only, so debt funds are expected to be missing there.


## 4. Data quality summary

In [6]:
nav = data["nav_history"]; nav["date"] = pd.to_datetime(nav["date"])
real = nav[nav["is_filled"] == 0] if "is_filled" in nav.columns else nav
print(f"NAV history: {real['date'].min():%d %b %Y} to {real['date'].max():%d %b %Y}, {real.groupby('amfi_code').size().iloc[0]} trading days per fund, {(nav['nav'] <= 0).sum()} non-positive NAVs")
tx = data["investor_transactions"]
print(f"Transactions: {len(tx):,} rows, {tx['investor_id'].nunique():,} investors, types: {tx['transaction_type'].value_counts().to_dict()}")
print(f"Expense ratio range in fund_master: {fm['expense_ratio_pct'].min():.2f}% to {fm['expense_ratio_pct'].max():.2f}% (expected 0.1 to 2.5)")

NAV history: 03 Jan 2022 to 29 May 2026, 1150 trading days per fund, 0 non-positive NAVs
Transactions: 32,778 rows, 5,000 investors, types: {'SIP': 19716, 'Lumpsum': 8095, 'Redemption': 4967}
Expense ratio range in fund_master: 0.55% to 1.64% (expected 0.1 to 2.5)
